In [3]:
!pip install duckdb pandas requests

In [4]:
import requests
import duckdb
import pandas as pd
from datetime import datetime

print("🔄 [Extract] A aceder às taxas de câmbio globais via API...")
url = "https://open.er-api.com/v6/latest/USD"
response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Falha ao buscar dados: {response.status_code}")

data = response.json()
rates = data.get("rates", {})

# Utilizando o Pandas perfeitamente, sem nenhum erro de DLL do Windows
df = pd.DataFrame(list(rates.items()), columns=["currency", "rate_to_usd"])
df["extracted_at"] = datetime.utcnow()
df["base_currency"] = "USD"

print("🔄 [Load] A guardar os dados tratados no Data Warehouse (DuckDB)...")

# Conectando ao DuckDB na nuvem do Colab
conn = duckdb.connect(database="warehouse.duckdb", read_only=False)

conn.execute("CREATE SCHEMA IF NOT EXISTS raw;")
conn.execute("DROP TABLE IF EXISTS raw.exchange_rates_raw;")
conn.execute("CREATE TABLE raw.exchange_rates_raw AS SELECT * FROM df;")

result = conn.execute("SELECT COUNT(*) FROM raw.exchange_rates_raw;").fetchone()[0]
print(f"✅ [Load] Sucesso! {result} linhas carregadas na tabela raw.exchange_rates_raw.")

conn.close()

🔄 [Extract] A aceder às taxas de câmbio globais via API...
🔄 [Load] A guardar os dados tratados no Data Warehouse (DuckDB)...
✅ [Load] Sucesso! 166 linhas carregadas na tabela raw.exchange_rates_raw.


/tmp/ipykernel_548/3321224636.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df["extracted_at"] = datetime.utcnow()
